# Exercise 01 — your first Kafka producer

**Goal:** send a single event into Redpanda, verify it arrived in three ways (delivery callback, Redpanda Console, partition+offset numbers), and explore what happens when things go wrong.

**Use case:** IoT sensors in several houses report electricity (`strom`) and water (`wasser`) consumption. Each event is keyed by house id (`haus_a`, `haus_b`, …) so events from the same house land in the same partition.

**Stuck?** Compare with [`../02_demo/demo_produce.ipynb`](../02_demo/demo_produce.ipynb). Full reference solution in [`../04_solution/exercise_01_produce_single.ipynb`](../04_solution/exercise_01_produce_single.ipynb).

## Step 1 — connect to the broker

**Concept check.** A *broker* is the server program that stores Kafka messages. Producers and consumers connect to it over TCP. Even if a real cluster has many brokers, you only need to give the producer *one* address as `bootstrap.servers` — the broker then teaches the producer about the rest of the cluster automatically.

Inside our docker network the broker is at `redpanda:29092`. The Codespace's port-forward `localhost:9092` would *also* work from your host, but **not** from another container — that's why all our notebooks use the internal address.

In [ ]:
from confluent_kafka import Producer
import json, time

BROKER = 'redpanda:29092'   # internal listener inside the docker network

conf = {
    'bootstrap.servers': BROKER,
    'client.id':         'python-producer',  # shows up in broker logs
}
producer = Producer(conf)
print(f'Producer connected to {BROKER}')

## Step 2 — build the event

Every Kafka message has three pieces:

| Piece | Purpose | Example |
|---|---|---|
| **Topic** | the *category* — like a folder | `strom` |
| **Key**   | routes the event to a partition | `haus_a` |
| **Value** | the payload (any bytes) | `'{"wert": 42.5, ...}'` |

**Why does the key matter?** Kafka splits each topic into *partitions* for parallelism. The producer hashes the key and uses `hash(key) % num_partitions` to decide which one. Same key → same partition → messages with that key arrive at the consumer **in order**. No key → round-robin → no ordering guarantee.

**Task — fill in two missing values below.**

In [ ]:
topic_name    = 'strom'
message_key   = 'haus_a'
message_value = json.dumps({
    'sensor':    'strom',
    'haus':      'haus_a',
    'wert':      None,    # TODO: pick a number, e.g. 42.5
    'einheit':   None,    # TODO: which unit fits electricity? ('kWh')
    'timestamp': time.time(),
})

print(f'Topic:  {topic_name}')
print(f'Key:    {message_key}')
print(f'Value:  {message_value}')

## Step 3 — send the event

`producer.produce(...)` doesn't actually send anything yet — it puts the message into an in-memory queue and returns immediately. The real send happens in a background thread. That's good for throughput (thousands of messages/second) but means we have to be careful at shutdown.

Two follow-ups:

- **`callback`** — Kafka calls this function once the broker has   confirmed (or rejected) each message. Useful for logging failures.
- **`flush()`** — blocks until the queue is empty. *Always* call this   before the program exits, otherwise messages can be silently lost.

**Task — call `producer.produce(...)` and `producer.flush()`.** Both key and value must be `bytes`, so call `.encode('utf-8')` on the strings.

In [ ]:
def delivery_report(err, msg):
    """Called by Kafka once delivery is confirmed (or has failed)."""
    if err:
        print(f'Delivery failed: {err}')
    else:
        print(f'Delivered to {msg.topic()} '
              f'[partition {msg.partition()}] offset {msg.offset()}')

# TODO: call producer.produce() with topic_name, message_key,
#       message_value (both .encode()'d) and callback=delivery_report


# TODO: call producer.flush() — why is this needed?
print('Done.')

**Verify it worked** in three places:

1. The `Delivered to strom [partition X] offset Y` line above.
2. **Redpanda Console** (port `8080` in *PORTS*) → *Topics* → `strom` →    *Messages*. Your event is at the bottom.
3. From a terminal: `rpk topic consume strom -X brokers=redpanda:29092    --num 1`.

## Task A — second key, observe the partition change

Send another event to `strom`, but with key `haus_b` this time. Compare the *partition* numbers in the output:

- Same partition as `haus_a`? Lucky hash collision; try `haus_c` too.
- Different partition? That's the normal case — keys are spread across   partitions, but each individual key always sticks to one.

Re-run the cell several times: the partition for `haus_b` *never* changes. Re-create the topic with a different number of partitions and the mapping changes — that's why partition counts are usually fixed at topic creation time.

In [ ]:
# TODO: build message_value_b for haus_b (similar to above)
# TODO: producer.produce(...) with key='haus_b', then flush


## Task B — write to a different topic

Send an event to topic `wasser` with key `haus_a` and unit `'Liter'`. This shows that **topics are independent**: same key in two different topics may land on completely different partition numbers, and consumers subscribe per topic.

In [ ]:
# TODO: send an event to topic 'wasser' for haus_a, einheit='Liter'


## Task C — what happens when the broker is unreachable?

Run the cell below. It points the producer at a non-existent address, so delivery will *fail*.

**Watch for two things:**

1. The error message — it tells you exactly why delivery failed.
2. The duration — `flush(timeout=5)` waits up to 5 s. Without the    `message.timeout.ms` setting, librdkafka would retry for **5 minutes**    by default.

**Think about it:** in a production system, what would you do with messages that fail to deliver? Drop them? Retry forever? Persist them to a separate "dead-letter" topic for inspection? — there is no single right answer; it depends on whether the data is recoverable and whether duplicates are acceptable.

In [ ]:
errors = []
def error_cb(err, msg):
    errors.append(str(err))

p_broken = Producer({
    'bootstrap.servers':  'localhost:9999',   # nothing listens here
    'message.timeout.ms': 4000,               # give up after 4 s
    'socket.timeout.ms':  2000,
})
p_broken.produce('strom', key=b'haus_a', value=b'{"test": true}',
                 callback=error_cb)
p_broken.flush(timeout=5)

print(errors[0] if errors else 'Delivered (unexpected)')

## What you learned

- The three pieces of a Kafka message: **topic, key, value**.
- Why the key matters: it determines the partition, and partitions   preserve ordering.
- Why we always call `producer.flush()` before exiting.
- That delivery is *asynchronous* and we use a callback to learn   about success or failure.
- That broker failures don't crash the producer; they show up via   the callback.

Next: read what you produced. Continue with [`exercise_02_consume_basic.ipynb`](exercise_02_consume_basic.ipynb).